# 🎬 Mazinger Studio

**Dub any video into another language with AI** — powered by [Mazinger](https://pypi.org/project/mazinger/).

Paste a YouTube URL (or upload a file), pick a language and voice, then hit **Start Dubbing**.

### How to use

1. **Run Step 1** below to install dependencies (~2 min)
2. **Run Step 2** to download & prepare models (~5-10 min first time)
3. **Run Step 3** to launch the app — a public link will appear
4. **Open the link** and start dubbing!

### What you need

| Requirement | Details |
|---|---|
| **GPU runtime** *(recommended)* | For voice synthesis — *Runtime → Change runtime type → T4 GPU* |
| **OpenAI API key** *(optional)* | Only needed if you choose OpenAI as LLM provider. By default we use **Ollama** (local, free). |

### Supported languages

Chinese, English, French, German, Italian, Japanese, Korean, Portuguese, Russian, Spanish

### Voice options

- **Voice Themes** — 16 pre-defined voice styles (narrator, young, deep, warm, news, storyteller, kid, teen — male & female). No files needed!
- **Preset Voices** — Clone from pre-built HuggingFace profiles
- **Custom Voice** — Upload your own 10-30 sec audio clip

In [ ]:
#@title 📦 Step 1: Install Dependencies { display-mode: "form" }

import subprocess, sys, shutil, time

# ── Ensure pip is available ──────────────────────────────────────
try:
    subprocess.check_call([sys.executable, "-m", "pip", "--version"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except subprocess.CalledProcessError:
    print("📥 Bootstrapping pip...")
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"],
                          stdout=subprocess.DEVNULL)

# ── Python packages ──────────────────────────────────────────────
print("Installing Mazinger with Qwen TTS + Faster Whisper...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                        "mazinger[tts,transcribe-faster]", "gradio>=4.0"])


In [ ]:
#@title 📥 Step 2: Download & Prepare Models { display-mode: "form" }

#@markdown Downloads all required models to disk so they're ready when needed.
#@markdown This avoids long waits during your first dubbing run.
#@markdown
#@markdown **Expected download sizes:**
#@markdown - Ollama LLM model: ~1.5 GB
#@markdown - Faster-Whisper (large-v3): ~3 GB
#@markdown - Qwen TTS: ~3-4 GB

import subprocess, sys, shutil, time
import base64, io

# ── Ollama (local LLM server) ───────────────────────────────────
if not shutil.which("ollama"):
    print("\n📥 Installing Ollama (local LLM server)...")
    subprocess.check_call(
        ["bash", "-c", "curl -fsSL https://ollama.com/install.sh | sh"],
        stdout=subprocess.DEVNULL,
    )
    print("✅ Ollama installed")
else:
    print("\n✅ Ollama already installed")

# Start Ollama server in background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
print("✅ Ollama server started (model will be pulled on first run)")

# ── GPU check ────────────────────────────────────────────────────
try:
    import torch
    if torch.cuda.is_available():
        print(f"\n✅ GPU detected: {torch.cuda.get_device_name(0)}")
    else:
        print("\n⚠️  No GPU detected — voice synthesis will be slow.")
        print("   Tip: Runtime → Change runtime type → T4 GPU")
except ImportError:
    pass

print("\n✅ Ready! Run the next cell to download & prepare models.")

print("📥 Downloading models (this may take 5-10 min on first run)...\n")

# ── 1. Ollama LLM model ─────────────────────────────────────────
print("1️⃣  Pulling Ollama LLM model (qwen3.5:2b-q8_0)...")
try:
    result = subprocess.run(
        ["ollama", "pull", "qwen3.5:2b-q8_0"],
        timeout=600,
    )
    if result.returncode == 0:
        print("   ✅ Ollama model ready\n")
    else:
        print("   ⚠️  Ollama pull had issues (model will be pulled on first run)\n")
except Exception as e:
    print(f"   ⚠️  Ollama pull failed: {e}\n")

# ── 2. Faster-Whisper model ─────────────────────────────────────
print("2️⃣  Downloading Faster-Whisper model (large-v3)...")
try:
    from huggingface_hub import snapshot_download
    snapshot_download("Systran/faster-whisper-large-v3")
    print("   ✅ Faster-Whisper model ready\n")
except Exception as e:
    print(f"   ⚠️  Faster-Whisper download failed: {e}\n")

# ── 3. Qwen TTS model ───────────────────────────────────────────
print("3️⃣  Downloading Qwen TTS model...")
try:
    from huggingface_hub import snapshot_download
    snapshot_download("Qwen/Qwen3-TTS-12Hz-1.7B-Base")
    print("   ✅ Qwen TTS model ready\n")
except Exception as e:
    print(f"   ⚠️  Qwen TTS download failed: {e}\n")

# ── 4. Qwen VoiceDesign model (for voice themes) ────────────────
print("4️⃣  Downloading Qwen VoiceDesign model (for voice themes)...")
try:
    from huggingface_hub import snapshot_download
    snapshot_download("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign")
    print("   ✅ Qwen VoiceDesign model ready\n")
except Exception as e:
    print(f"   ⚠️  Qwen VoiceDesign download failed: {e}\n")

# ── 5. Warm up Ollama (load model into GPU) ──────────────────────
print("5️⃣  Warming up Ollama LLM (loading model into GPU)...")
try:
    import json, urllib.request
    _body = json.dumps({
        "model": "qwen3.5:2b-q8_0",
        "messages": [{"role": "user", "content": "Reply with: ready"}],
        "stream": False,
        "think": False,
        "options": {"temperature": 0.1},
    }).encode()
    _req = urllib.request.Request(
        "http://localhost:11434/api/chat", _body,
        headers={"Content-Type": "application/json"},
    )
    t0 = time.time()
    with urllib.request.urlopen(_req, timeout=120) as _resp:
        _result = json.loads(_resp.read())
    _reply = _result.get("message", {}).get("content", "")
    _tokens = _result.get("eval_count", 0)
    print(f"   ✅ Ollama responded in {time.time() - t0:.1f}s ({_tokens} tokens): {_reply[:50]}\n")
except Exception as e:
    print(f"   ⚠️  Warm-up failed: {e}\n")

print("✅ All models downloaded and cached! Run the next cell to launch the app.")

# ── Benchmark Ollama LLM with test image ─────────────────────────




# ── Create a tiny 64×64 red test image ───────────────────────────
try:
    from PIL import Image as _PILImage
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "Pillow"])
    from PIL import Image as _PILImage

_img = _PILImage.new("RGB", (64, 64), color=(200, 60, 60))
_buf = io.BytesIO()
_img.save(_buf, format="JPEG", quality=70)
_b64 = base64.b64encode(_buf.getvalue()).decode()

# ── Build mazinger LLM client (native Ollama, thinking disabled) ─
from mazinger.llm import build_client

_client = build_client(
    base_url="http://localhost:11434/v1",
    think=False,
)

_model = "qwen3.5:2b-q8_0"
_messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe this image in one sentence."},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{_b64}"}},
        ],
    }
]

# ── Run benchmark ────────────────────────────────────────────────
print(f"🏎️  Benchmarking {_model} with a 64×64 test image…\n")

_t0 = time.time()
_resp = _client.chat.completions.create(model=_model, messages=_messages, temperature=0.1)
_elapsed = time.time() - _t0

_text = _resp.choices[0].message.content
_prompt_tok = _resp.usage.prompt_tokens
_gen_tok = _resp.usage.completion_tokens
_tok_per_sec = _gen_tok / _elapsed if _elapsed > 0 else 0

print(f"   Response   : {_text[:120]}")
print(f"   Prompt tok : {_prompt_tok}")
print(f"   Gen tokens : {_gen_tok}")
print(f"   Wall time  : {_elapsed:.2f}s")
print(f"   Speed      : {_tok_per_sec:.1f} tok/s")
print()

if _tok_per_sec < 5:
    print("⚠️  Throughput is low — consider a smaller model or check GPU availability.")
elif _tok_per_sec < 20:
    print("✅ Reasonable speed. Dubbing will work but may be a bit slow on long videos.")
else:
    print("🚀 Great throughput! You're ready to dub.")


In [ ]:
#@title 🚀 Step 3: Launch Mazinger Studio { display-mode: "form" }

import subprocess, os

# ── Fetch studio scripts from GitHub ─────────────────────────────
_BASE = "https://raw.githubusercontent.com/bakrianoo/mazinger/refs/heads/master/docs/notebooks/studio"
_SCRIPTS = ["constants.py", "theme.py", "helpers.py", "pipeline.py", "app.py"]

os.makedirs("studio", exist_ok=True)
for _script in _SCRIPTS:
    _dest = os.path.join("studio", _script)
    if not os.path.exists(_dest):
        subprocess.check_call(["wget", "-q", f"{_BASE}/{_script}", "-O", _dest])
        print(f"  ✅ Downloaded {_script}")
    else:
        print(f"  ✔ {_script} (cached)")

print("\n🚀 Launching Mazinger Studio...\n")

# ── Run the app ──────────────────────────────────────────────────
subprocess.check_call(["python", "studio/app.py"])